In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew
import plotly.express as px
import plotly.graph_objects as go

### 特徴量性格診断

In [ ]:
# --- ファイル読み込み ---

df_feats_d = pd.read_parquet("liq_engine_feats_d.parquet", engine="pyarrow")
df_feats_w = pd.read_parquet("liq_engine_feats_w.parquet", engine="pyarrow")
df_feats_w.columns

In [ ]:
df_feats_w["DXF_roc4"] = df_feats_w["DX-Y.NYB"].pct_change(4)
target_return = 'forward_8w_return'
df_feats_w[target_return] = df_feats_w["^GSPC"].shift(-8) / df_feats_w["^GSPC"] -1

In [ ]:
df = df_feats_w.copy()
indicators = ['Liq_eff', 'NFCI_z52', 'DFII10', 'DXF_roc4', 'Cu_Au_Ratio_z52']

for ind in indicators:
    is_inverse = True if ind in ['NFCI', 'DFII10', 'DXF_roc4', 'CP_Spread'] else False
    df[f'{ind}_Regime'] = pd.qcut(df[ind], 5, labels=False, duplicates='drop') + 1
    if is_inverse:
        df[f'{ind}_Regime'] = 6 - df[f'{ind}_Regime'] # 1が最悪(重力)、5が最高(浮力)に統一

stats_list = []
for ind in indicators:
    regime_col = f'{ind}_Regime'
    for r in range(1, 6):
        subset = df[df[regime_col] == r][target_return].dropna()
        # CVaR (期待ショートフォール) : 下位5%の平均リターン
        cvar_5 = subset[subset <= subset.quantile(0.05)].mean()
        stats_list.append({
            'Indicator': ind,
            'Regime': r,
            'Skewness': skew(subset),
            'CVaR_5%': cvar_5,
            'Mean': subset.mean()
        })

stats_df = pd.DataFrame(stats_list)
stats_df

In [ ]:
fig = px.bar(
    stats_df[stats_df['Regime'] == 1], 
    x='Indicator', y='CVaR_5%', 
    title="Regime 1 (Heavy Gravity) comparison: CVaR 5% (Bottom Tail Risk)",
    labels={'CVaR_5%': 'Average Return of Bottom 5% Cases'},
    color='Indicator',
    text_auto='.1%'
)
fig.update_layout(yaxis_title="Potential Crash Depth (CVaR)", template="plotly_white")
fig.show(renderer="browser",config=dict(displayModeBar=False))

# 4. 可視化：Skewness（歪度）の比較
fig_skew = px.line(
    stats_df, x='Regime', y='Skewness', color='Indicator', markers=True,
    title="Skewness Transition: From Gravity (1) to Buoyancy (5)",
    labels={'Skewness': 'Distribution Skewness (Negative = Left Tail)'}
)
fig_skew.add_hline(y=0, line_dash="dash")
fig_skew.show(renderer="browser",config=dict(displayModeBar=False))

In [22]:
df_feats_w = pd.read_parquet("liq_engine_feats.parquet", engine="pyarrow")
#df_feats_w = pd.read_parquet("liq_engine_feats_w.parquet", engine="pyarrow")
df_feats_w.columns

Index(['^GSPC', 'TLT', 'BAMLH0A0HYM2', 'RRPONTSYD', 'DTB3', 'DFF', 'DX-Y.NYB',
       'HG=F', 'GC=F', 'DFII10', 'THREEFYTP10', 'UUP', 'WALCL', 'WDTGAL',
       'WCURCIR', 'NFCI', 'TOTBKCR', 'M2SL', 'INDPRO', 'NDFACBM027SBOG',
       'BUSLOANS', 'RRP_filled', 'Net_Liquidity', 'Net_Liquidity_z252',
       'NFCI_z252', 'Liq_eff', 'Liq_eff_ma5', 'Liq_eff_diff20',
       'Marshallian_K', 'Credit_Growth_z252', 'Cu_Au_Ratio_z252', 'UUP_z52',
       'NDFACBM027SBOG_z52', 'Real_Rate_Gravity', 'Term_Premium',
       'HY_Spread_Momentum', 'DTB3_DFF_Spread', 'DXY_roc65'],
      dtype='str')

In [23]:

#df_feats_d["DXY_roc20"] = df_feats_d["DX-Y.NYB"].pct_change(20)
target = "Liq_eff_diff20"
df = df_feats_w[["^GSPC", target]].dropna()
forward_weeks = 13
df['forward_return'] = df['^GSPC'].shift(-forward_weeks) / df['^GSPC'] - 1

# 3. Liq_eff を5つのクオンタイルに分割
# Z-scoreの絶対値で分けるよりも、データ数を揃えた方が統計的な比較がしやすいため
df['Liq_Regime'] = pd.qcut(df[target], 5, 
                          labels=['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)'])

# 4. ナン値を削除（最後の8週間分はリターンが計算できないため）
plot_df = df.dropna(subset=['forward_return', 'Liq_Regime'])

# 5. Plotlyで箱ひげ図を作成
fig = px.box(
    plot_df, 
    x='Liq_Regime', 
    y='forward_return',
    color='Liq_Regime',
    points="all", # 全データ点（ジッター）を表示して密度を確認
    title=f"Market Physics: S&P 500 {forward_weeks}W Forward Return by Liq_eff Regime",
    labels={'forward_return': f'{forward_weeks}-Week Forward Return', 'Liq_Regime': 'Liquidity Regime'},
    category_orders={"Liq_Regime": ['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)']}
)

# 0%のライン（損益分岐）を強調
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Break Even")

# レイアウト調整
fig.update_layout(
    xaxis_title="Liquidity Regime (Liq_eff Z-score)",
    yaxis_title=f"Forward {forward_weeks}W Return (%)",
    showlegend=False,
    template="plotly_white",
    yaxis=dict(tickformat=".1%")
)

# 表示
fig.show()

# --- 統計的な裏付け（Skewness）の表示 ---
print("--- Statistical Skewness by Regime ---")
for regime in df['Liq_Regime'].unique():
    if pd.isna(regime): continue
    skew = plot_df[plot_df['Liq_Regime'] == regime]['forward_return'].skew()
    print(f"{regime}: Skewness = {skew:.2f}")

--- Statistical Skewness by Regime ---
5: Very High (Buoyancy): Skewness = -1.58
4: High: Skewness = -1.34
3: Neutral: Skewness = 0.53
1: Very Low (Heavy): Skewness = 0.22
2: Low: Skewness = -0.39
